In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV
from sklearn.pipeline import Pipeline

In [2]:
df = pd.read_csv('../data/filtered/Fil_SCF_20022.csv', index_col=0)

In [3]:
df.columns

Index(['NETWORTH', 'INCOME', 'FIN', 'NFIN', 'DEBT', 'EQUITY', 'STOCKS',
       'HHOUSES', 'BUS', 'RETQLIQ', 'YESFINRISK', 'SPENDMOR', 'LATE',
       'EMERGSAV', 'LIQ', 'FINLIT', 'AGE', 'EDUC', 'MARRIED', 'KIDS', 'OCCAT1',
       'RET_ASSETS'],
      dtype='str')

Transform target 

In [4]:
target = 'RET_ASSETS'
log_target = 'LOG_RET_ASSETS'

In [5]:
df[log_target] = np.log1p(df[target])
df = df.drop([target], axis=1)
df.shape

(4595, 22)

In [6]:
corr = df.corr(numeric_only=True)
corr[log_target].sort_values(ascending=False)

LOG_RET_ASSETS    1.000000
EDUC              0.572694
HHOUSES           0.530207
EMERGSAV          0.516679
FINLIT            0.416367
AGE               0.317108
NETWORTH          0.313487
RETQLIQ           0.268639
NFIN              0.265369
FIN               0.251892
BUS               0.236071
INCOME            0.221289
EQUITY            0.210079
LIQ               0.169290
DEBT              0.162331
STOCKS            0.142853
YESFINRISK        0.021303
OCCAT1           -0.014251
SPENDMOR         -0.017338
KIDS             -0.106684
LATE             -0.227441
MARRIED          -0.378186
Name: LOG_RET_ASSETS, dtype: float64

In [7]:
X = df.drop(columns=[log_target])
y = df[log_target]

model = RandomForestRegressor(random_state=42)
model.fit(X, y)

importance = pd.Series(model.feature_importances_, index=X.columns)
importance.sort_values(ascending=False)

FIN           9.959511e-01
RETQLIQ       3.228791e-03
LIQ           7.279389e-04
EQUITY        2.702579e-05
STOCKS        1.530544e-05
NETWORTH      9.159245e-06
INCOME        7.739040e-06
AGE           7.037208e-06
NFIN          5.210018e-06
DEBT          4.444220e-06
EDUC          3.040981e-06
SPENDMOR      2.744702e-06
BUS           2.211301e-06
KIDS          2.074913e-06
FINLIT        1.606254e-06
OCCAT1        1.412199e-06
MARRIED       1.096445e-06
EMERGSAV      9.218948e-07
YESFINRISK    4.205414e-07
LATE          3.936916e-07
HHOUSES       2.925049e-07
dtype: float64

In [8]:
result = permutation_importance(model, X, y, n_repeats=10, random_state=42)

perm_importance = pd.Series(result.importances_mean, index=X.columns)
perm_importance.sort_values(ascending=False)

FIN           1.829474e+00
RETQLIQ       8.528193e-03
LIQ           1.727116e-03
EQUITY        2.678541e-05
STOCKS        5.152492e-06
NETWORTH      3.057793e-06
INCOME        1.747814e-06
AGE           1.672401e-06
NFIN          7.779538e-07
DEBT          7.396362e-07
SPENDMOR      5.526160e-07
BUS           4.753268e-07
EDUC          4.248648e-07
MARRIED       3.861635e-07
KIDS          3.592048e-07
FINLIT        2.679932e-07
OCCAT1        2.065101e-07
YESFINRISK    1.484295e-07
EMERGSAV      1.315398e-07
LATE          5.839766e-08
HHOUSES       3.061580e-08
dtype: float64

In [9]:
lasso_cv_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("lasso_cv", LassoCV(cv=5, max_iter=100000, random_state=42))
])

lasso_cv_pipeline.fit(X, y)

lasso_cv = lasso_cv_pipeline.named_steps["lasso_cv"]

lasso_cv_coefficients = pd.Series(
    lasso_cv.coef_,
    index=X.columns
).sort_values(key=abs, ascending=False)

print("Best alpha:", lasso_cv.alpha_)
lasso_cv_coefficients

Best alpha: 0.03202480521168103


EDUC          1.302686
HHOUSES       0.865479
EMERGSAV      0.764154
MARRIED      -0.562841
NETWORTH      0.465213
AGE           0.462113
FINLIT        0.446335
FIN           0.254223
OCCAT1       -0.203624
RETQLIQ       0.165236
STOCKS       -0.157169
KIDS         -0.143071
LATE         -0.130575
DEBT          0.096155
SPENDMOR     -0.067595
YESFINRISK    0.063883
INCOME       -0.020946
BUS           0.000000
LIQ          -0.000000
EQUITY        0.000000
NFIN          0.000000
dtype: float64

In [10]:
vars_to_keep = [
    "LOG_RET_ASSETS",
    "INCOME",
    "DEBT",
    "LIQ",
    "EDUC",
    "HHOUSES",
    "EMERGSAV",
    "FINLIT",
    "AGE",
    "MARRIED",
    "KIDS",
    "OCCAT1",
    "LATE",
    "SPENDMOR",
    "YESFINRISK"
]

In [11]:
df_new = df[vars_to_keep]

In [12]:
corr = df_new.corr()
corr[log_target].sort_values(ascending=False)

LOG_RET_ASSETS    1.000000
EDUC              0.572694
HHOUSES           0.530207
EMERGSAV          0.516679
FINLIT            0.416367
AGE               0.317108
INCOME            0.221289
LIQ               0.169290
DEBT              0.162331
YESFINRISK        0.021303
OCCAT1           -0.014251
SPENDMOR         -0.017338
KIDS             -0.106684
LATE             -0.227441
MARRIED          -0.378186
Name: LOG_RET_ASSETS, dtype: float64

Correlation analysis showed that education, homeownership, emergency savings, financial literacy, and age had the strongest positive relationships with LOG_RET_ASSETS. Late payments and number of children showed negative relationships. These results suggest that retirement-relevant assets are associated with both financial behavior and demographic characteristics.

In [13]:
df_new.describe()

,LOG_RET_ASSETS,INCOME,DEBT,LIQ,EDUC,HHOUSES,EMERGSAV,FINLIT,AGE,MARRIED,KIDS,OCCAT1,LATE,SPENDMOR,YESFINRISK
count,4595.000000,4.595000e+03,4.595000e+03,4.595000e+03,4595.000000,4595.000000,4595.000000,4595.000000,4595.000000,4595.000000,4595.000000,4595.000000,4595.000000,4595.000000,4595.000000
mean,11.362162,1.593197e+06,3.653292e+05,7.702663e+05,10.327965,0.677040,0.492274,2.303591,54.468988,1.367791,0.738629,1.830686,0.113384,3.506638,0.053101
std,3.945446,1.233521e+07,2.630899e+06,8.239290e+06,2.816657,0.467658,0.499995,0.829303,16.190491,0.482257,1.108118,0.933612,0.317096,1.359158,0.224260
min,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,-1.000000,0.000000,0.000000,0.000000,18.000000,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000
25%,8.665786,4.215556e+04,0.000000e+00,1.935000e+03,8.000000,0.000000,0.000000,2.000000,42.000000,1.000000,0.000000,1.000000,0.000000,2.000000,0.000000
50%,11.813037,9.403933e+04,2.900000e+04,1.460000e+04,11.000000,1.000000,0.000000,3.000000,56.000000,1.000000,0.000000,2.000000,0.000000,4.000000,0.000000
75%,14.237899,2.648234e+05,2.162750e+05,9.960000e+04,12.000000,1.000000,1.000000,3.000000,67.000000,2.000000,1.000000,3.000000,0.000000,5.000000,0.000000
max,21.413611,4.532588e+08,1.238900e+08,2.676600e+08,14.000000,1.000000,1.000000,3.000000,95.000000,2.000000,10.000000,4.000000,1.000000,5.000000,1.000000


In [14]:
df_new['LOG_LIQ'] = np.log1p(df['LIQ'])
df_new['LOG_INCOME'] = np.log1p(df['INCOME'])
df_new['LOG_DEBT'] = np.log1p(df['DEBT'])

In [15]:
df_new.columns

Index(['LOG_RET_ASSETS', 'INCOME', 'DEBT', 'LIQ', 'EDUC', 'HHOUSES',
       'EMERGSAV', 'FINLIT', 'AGE', 'MARRIED', 'KIDS', 'OCCAT1', 'LATE',
       'SPENDMOR', 'YESFINRISK', 'LOG_LIQ', 'LOG_INCOME', 'LOG_DEBT'],
      dtype='str')

In [16]:
df_new.drop(columns=['LIQ', 'INCOME', 'DEBT'], inplace=True)

In [17]:
df_new.columns

Index(['LOG_RET_ASSETS', 'EDUC', 'HHOUSES', 'EMERGSAV', 'FINLIT', 'AGE',
       'MARRIED', 'KIDS', 'OCCAT1', 'LATE', 'SPENDMOR', 'YESFINRISK',
       'LOG_LIQ', 'LOG_INCOME', 'LOG_DEBT'],
      dtype='str')

In [18]:
df_new.describe()

,LOG_RET_ASSETS,EDUC,HHOUSES,EMERGSAV,FINLIT,AGE,MARRIED,KIDS,OCCAT1,LATE,SPENDMOR,YESFINRISK,LOG_LIQ,LOG_INCOME,LOG_DEBT
count,4595.000000,4595.000000,4595.000000,4595.000000,4595.000000,4595.000000,4595.000000,4595.000000,4595.000000,4595.000000,4595.000000,4595.000000,4595.000000,4595.000000,4595.000000
mean,11.362162,10.327965,0.677040,0.492274,2.303591,54.468988,1.367791,0.738629,1.830686,0.113384,3.506638,0.053101,9.406495,11.674989,8.154658
std,3.945446,2.816657,0.467658,0.499995,0.829303,16.190491,0.482257,1.108118,0.933612,0.317096,1.359158,0.224260,3.169348,2.007434,5.290533
min,0.000000,-1.000000,0.000000,0.000000,0.000000,18.000000,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000
25%,8.665786,8.000000,0.000000,0.000000,2.000000,42.000000,1.000000,0.000000,1.000000,0.000000,2.000000,0.000000,7.568349,10.649146,0.000000
50%,11.813037,11.000000,1.000000,0.000000,3.000000,56.000000,1.000000,0.000000,2.000000,0.000000,4.000000,0.000000,9.588845,11.451479,10.275086
75%,14.237899,12.000000,1.000000,1.000000,3.000000,67.000000,2.000000,1.000000,3.000000,0.000000,5.000000,0.000000,11.508927,12.486822,12.284310
max,21.413611,14.000000,1.000000,1.000000,3.000000,95.000000,2.000000,10.000000,4.000000,1.000000,5.000000,1.000000,19.405228,19.931974,18.634905


In [19]:
corr = df_new.corr(numeric_only=True)
corr[log_target].sort_values(ascending=False)

LOG_RET_ASSETS    1.000000
LOG_LIQ           0.875612
LOG_INCOME        0.627728
EDUC              0.572694
HHOUSES           0.530207
EMERGSAV          0.516679
FINLIT            0.416367
AGE               0.317108
LOG_DEBT          0.104704
YESFINRISK        0.021303
OCCAT1           -0.014251
SPENDMOR         -0.017338
KIDS             -0.106684
LATE             -0.227441
MARRIED          -0.378186
Name: LOG_RET_ASSETS, dtype: float64

In [21]:
df_new.columns

Index(['LOG_RET_ASSETS', 'EDUC', 'HHOUSES', 'EMERGSAV', 'FINLIT', 'AGE',
       'MARRIED', 'KIDS', 'OCCAT1', 'LATE', 'SPENDMOR', 'YESFINRISK',
       'LOG_LIQ', 'LOG_INCOME', 'LOG_DEBT'],
      dtype='str')

In [22]:
X = df_new.drop(columns=[log_target])
y = df_new[log_target]

model = RandomForestRegressor(random_state=42)
model.fit(X, y)

importance = pd.Series(model.feature_importances_, index=X.columns)
importance.sort_values(ascending=False)

LOG_LIQ       0.683693
LOG_INCOME    0.200714
AGE           0.034345
LOG_DEBT      0.024967
EDUC          0.014551
SPENDMOR      0.008072
FINLIT        0.007258
KIDS          0.005939
HHOUSES       0.005487
OCCAT1        0.005426
EMERGSAV      0.003553
MARRIED       0.002832
LATE          0.001727
YESFINRISK    0.001436
dtype: float64

In [29]:
lasso_cv_pipeline.fit(X, y)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('scaler', ...), ('lasso_cv', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"eps eps: float, default=1e-3Length of the path. ``eps=1e-3`` means that``alpha_min / alpha_max = 1e-3``.",0.001
,"n_alphas n_alphas: int, default=100Number of alphas along the regularization path... deprecated:: 1.7 `n_alphas` was deprecated in 1.7 and will be removed in 1.9. Use `alphas` instead.",'deprecated'
,"alphas alphas: array-like or int, default=NoneValues of alphas to test along the regularization path.If int, `alphas` values are generated automatically.If array-like, list of alpha values to use... versionchanged:: 1.7 `alphas` accepts an integer value which removes the need to pass `n_alphas`... deprecated:: 1.7 `alphas=None` was deprecated in 1.7 and will be removed in 1.9, at which point the default value will be set to 100.",'warn'
,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto false, no intercept will be used in calculations(i.e. data is expected to be centered).",True


In [24]:
lasso = lasso_cv_pipeline.named_steps['lasso_cv']
pd.Series(lasso.coef_, index=X.columns).sort_values()

OCCAT1       -0.150683
MARRIED      -0.117245
KIDS         -0.115729
SPENDMOR     -0.003835
LATE          0.000000
LOG_DEBT      0.002639
YESFINRISK    0.012409
EMERGSAV      0.180901
FINLIT        0.202229
AGE           0.324805
HHOUSES       0.348220
LOG_INCOME    0.485116
EDUC          0.488173
LOG_LIQ       2.434067
dtype: float64

Final features and target variable

In [25]:
final_features = [
    "LOG_LIQ",
    "EDUC",
    "LOG_INCOME",
    "HHOUSES",
    "AGE",
    "FINLIT",
    "EMERGSAV",
    "KIDS",
    "MARRIED",
    "OCCAT1",
    "YESFINRISK",
    "LOG_DEBT",
    "SPENDMOR",
    "LATE",
    log_target
]

df_final = df_new[final_features]

In [26]:
df_final.to_csv('../data/processed/Cleaned_SCF_2022.csv')

Confirm 

In [27]:
df_final.shape

(4595, 15)

In [28]:
df_final.head()

,LOG_LIQ,EDUC,LOG_INCOME,HHOUSES,AGE,FINLIT,EMERGSAV,KIDS,MARRIED,OCCAT1,YESFINRISK,LOG_DEBT,SPENDMOR,LATE,LOG_RET_ASSETS
0,9.480444,9,10.566323,1,70,1,1,2,2,3,0,12.180760,5,0,13.128546
5,10.609082,12,12.323103,1,46,3,1,0,2,2,0,13.159083,3,0,12.390480
10,9.295692,14,13.006586,1,68,3,1,0,1,3,0,16.682273,3,1,17.925595
15,12.007628,12,11.934327,0,74,3,1,0,2,3,0,0.000000,3,0,15.113520
20,7.003974,8,10.936822,0,19,0,0,0,1,1,0,9.305741,2,0,7.003974
